In [ ]:
import numpy as np
import scipy.stats as stats
import scipy.signal as signal
import torch
from jaxtyping import Float
import pandas as pd
import matplotlib.pyplot as plt

# custom utils

# pattern_lens

# attention-motifs
from attention_motifs.features import scalar_feature_table

In [ ]:
def vec_features(
	x: torch.Tensor,
	prefix: str | None = None,
	compute_distribution: bool = True,
	compute_timeseries: bool = True,
) -> dict[str, float]:
	if x.dim() != 1:
		raise ValueError("Input tensor x must be 1-dimensional")

	# Convert to a NumPy array (detaching from GPU if needed)
	arr: np.ndarray = x.detach().cpu().numpy()
	n: int = arr.size

	dist_features: dict[str, float] = dict()
	timeseries_features: dict[str, float] = dict()

	if compute_distribution:
		bins: int = 10 if n >= 10 else n
		hist, _ = np.histogram(arr, bins=bins)
		probs: np.ndarray = (
			hist.astype(float) / hist.sum() if hist.sum() > 0 else hist.astype(float)
		)
		dist_features = dict(
			mean=np.mean(arr),
			median=np.median(arr),
			variance=np.var(arr, ddof=1),
			std=np.std(arr, ddof=1),
			skewness=stats.skew(arr),
			kurtosis=stats.kurtosis(arr),
			entropy=stats.entropy(probs, base=2),
			L1_norm=np.sum(np.abs(arr)) / n,
			L2_norm=np.linalg.norm(arr, ord=2) / n,
			rms=np.sqrt(np.mean(arr**2)),
			energy=np.sum(arr**2),
		)

	if compute_timeseries:
		# Lag-1 Autocorrelation (Pearson correlation between arr[:-1] and arr[1:])
		autocorr_lag1: float = (
			np.corrcoef(arr[:-1], arr[1:])[0, 1]
			if np.std(arr[:-1]) > 0 and np.std(arr[1:]) > 0
			else 0.0
		)

		# PSD using Welch's method (total power)
		freqs, psd_vals = signal.welch(arr, nperseg=n)
		psd_total_power: float = np.sum(psd_vals)

		# Linear regression using scipy.stats.linregress
		t: np.ndarray = np.arange(n)
		linreg_result = stats.linregress(t, arr)
		line_fit: dict[str, float] = dict(
			slope=linreg_result.slope,
			intercept=linreg_result.intercept,
			r2=linreg_result.rvalue**2,
		)

		timeseries_features: dict[str, float] = dict(
			zero_crossing_rate=np.sum(np.diff(np.signbit(arr))) / (n - 1),
			autocorr_lag1=autocorr_lag1,
			psd_total_power=psd_total_power,
			**{f"linreg.{k}": v for k, v in line_fit.items()},
		)

	output: dict[str, float] = {**dist_features, **timeseries_features}
	if prefix:
		output = {f"{prefix}.{k}": v for k, v in output.items()}
	return output


def compute_scalar_features(
	attn_batch: Float[torch.Tensor, "n_ctx n_ctx"],
) -> dict[str, float]:
	return dict(
		**vec_features(attn_batch.diagonal(), prefix="diag"),
		**vec_features(attn_batch[:, 0], prefix="first_tok"),
	)


df: pd.DataFrame = scalar_feature_table(features_func=compute_scalar_features)

In [ ]:
# jsonl_write(
# 	"../data/scalar_features.jsonl.gz", df.to_dict(orient="records"), use_gzip=True
# )

In [ ]:
# df = pd.DataFrame(jsonl_load("../data/scalar_features.jsonl.gz"))

In [ ]:
def plot_histograms_long(df: pd.DataFrame) -> None:
	"""Plot histograms for each feature with different models superimposed.

	This function assumes the DataFrame is in long format with columns:
	"model", "feat_name", "feat_val", and optionally "prompt", "layer", "head".

	# Parameters:
	 - `df : pd.DataFrame`
		 DataFrame containing the data.

	# Returns:
	 - `None`
		 Displays the histograms.
	"""
	features: list[str] = df["feat_name"].unique().tolist()
	models: list[str] = df["model"].unique().tolist()

	for feature in features:
		plt.figure()
		subset_feature: pd.DataFrame = df[df["feat_name"] == feature]
		for model in models:
			subset_model: pd.DataFrame = subset_feature[
				subset_feature["model"] == model
			]
			plt.hist(
				subset_model["feat_val"], bins=50, alpha=0.5, label=model, density=True
			)
		plt.xlabel(feature)
		plt.ylabel("Frequency")
		plt.title(f"Histogram of {feature} for different models")
		plt.legend()
		plt.show()


plot_histograms_long(df)